# 16b_score_finbert.ipynb

**Purpose:** Score the 796,038 scoring-input headlines with FinBERT (`ProsusAI/finbert`, Araci 2019) on a Colab T4 GPU, producing per-headline class probabilities `p_pos`, `p_neu`, `p_neg`.

**Inputs:** `/content/drive/MyDrive/thesis_finbert/finbert_input.csv` — uploaded from `Data/colab/finbert_input.csv`, which `pipeline/16a_export_for_colab.py` produces; SHA256 and row count are asserted on load.

**Outputs:** `/content/drive/MyDrive/thesis_finbert/finbert_scores_partial.csv` (per-chunk checkpoint) and `/content/drive/MyDrive/thesis_finbert/finbert_scores.csv` (final, ordered by `hid`). The final file is copied back to `Data/colab/finbert_scores.csv` and consumed by `pipeline/16c_finbert_aggregate_freeze.py`.

**Notes**
- Runtime: T4 GPU (Runtime → Change runtime type); fp16 inference with fp32 softmax; run the cells top to bottom.
- Resume: the full run appends to the partial file every 50,000 rows; re-running the notebook skips the `hid`s already scored.
- Model: `ProsusAI/finbert` (not `finbert-tone`); the label-to-logit mapping is read by name from `model.config.id2label`.
- Batch sizing: 256 headlines per batch, max 64 tokens; halve `BATCH` on OOM.
- `hid` is the row position assigned by 16a and the join key used by 16c.

In [ ]:
# Step 1: environment, mount Drive, verify the input file (SHA256, row count, hid range)
import hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch, transformers

print('torch', torch.__version__, '| transformers', transformers.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/thesis_finbert')   # upload finbert_input.csv here
INPUT, PARTIAL, FINAL = BASE / 'finbert_input.csv', BASE / 'finbert_scores_partial.csv', BASE / 'finbert_scores.csv'

INPUT_SHA = 'e87ba39d4f43081736e03d9fa4a9e89df54658e3d08dea3e48d51ed01ebef346'
N_EXPECTED = 796_038

sha = hashlib.sha256(INPUT.read_bytes()).hexdigest()
assert sha == INPUT_SHA, f'input SHA mismatch: {sha} - re-upload the file from 16a'
inp = pd.read_csv(INPUT, dtype={'hid': 'int64', 'PERMNO': 'int64', 'target_week': str, 'headline': str})
assert len(inp) == N_EXPECTED and inp['hid'].is_unique
assert inp['hid'].min() == 0 and inp['hid'].max() == N_EXPECTED - 1
print(f'input OK: {len(inp):,} rows, SHA verified {sha[:8]}...')

In [ ]:
# Step 2: load ProsusAI/finbert; map label names to logit indices from model.config.id2label
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = 'ProsusAI/finbert'          # Araci (2019) - locked identity
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.eval().half().cuda()             # fp16 inference; softmax done in fp32

print('model.config.id2label:', model.config.id2label)
id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
assert set(id2label.values()) == {'positive', 'negative', 'neutral'}, id2label
IDX = {name: i for i, name in id2label.items()}   # label NAME -> logit index
print('mapped by NAME -> logit index:', IDX)

In [ ]:
# Step 3: smoke test on five headlines; the clear positive/negative cases must pass before the full run
def score_texts(texts, max_length=64):
    enc = tok(texts, truncation=True, max_length=max_length, padding=True,
              return_tensors='pt').to('cuda')
    with torch.no_grad():
        logits = model(**enc).logits.float()          # fp32 softmax
    p = torch.softmax(logits, dim=-1).cpu().numpy()
    return p[:, IDX['positive']], p[:, IDX['neutral']], p[:, IDX['negative']]

smoke = ['Company beats earnings estimates, raises guidance',
         'Company files for bankruptcy protection',
         'Company announces quarterly dividend',
         inp['headline'].iloc[0],
         inp['headline'].iloc[500000]]
pp, pu, pn = score_texts(smoke)
for t, a, b, c in zip(smoke, pp, pu, pn):
    print(f'p_pos {a:.3f}  p_neu {b:.3f}  p_neg {c:.3f}  | {t[:90]}')
assert pp[0] > max(pu[0], pn[0]), 'SMOKE FAIL: beats-guidance not positive'
assert pn[1] > max(pp[1], pu[1]), 'SMOKE FAIL: bankruptcy not negative'
print('SMOKE TEST PASSED - proceed to the full run')

In [ ]:
# Step 4: full run in 50k-row chunks, appended to the Drive partial file after each chunk (resumable by hid)
CHUNK, BATCH, MAX_LEN = 50_000, 256, 64      # halve BATCH if OOM

done = set()
if PARTIAL.exists():
    done = set(pd.read_csv(PARTIAL, usecols=['hid'])['hid'].tolist())
    print(f'resuming: {len(done):,} hids already scored')
todo = inp[~inp['hid'].isin(done)].reset_index(drop=True)
print(f'to score: {len(todo):,}')

n_trunc, t0 = 0, time.time()
for c0 in range(0, len(todo), CHUNK):
    sub = todo.iloc[c0:c0 + CHUNK]
    out = np.empty((len(sub), 3), dtype=np.float32)
    for b0 in range(0, len(sub), BATCH):
        texts = sub['headline'].iloc[b0:b0 + BATCH].tolist()
        lens = tok(texts, truncation=False, padding=False)['input_ids']
        n_trunc += sum(len(x) > MAX_LEN for x in lens)
        enc = tok(texts, truncation=True, max_length=MAX_LEN, padding=True,
                  return_tensors='pt').to('cuda')
        with torch.no_grad():
            logits = model(**enc).logits.float()
        p = torch.softmax(logits, dim=-1).cpu().numpy()
        out[b0:b0 + len(texts), 0] = p[:, IDX['positive']]
        out[b0:b0 + len(texts), 1] = p[:, IDX['neutral']]
        out[b0:b0 + len(texts), 2] = p[:, IDX['negative']]
    chunk_df = pd.DataFrame({'hid': sub['hid'].to_numpy(),
                             'p_pos': out[:, 0], 'p_neu': out[:, 1], 'p_neg': out[:, 2]})
    chunk_df.to_csv(PARTIAL, mode='a', header=not PARTIAL.exists(), index=False)
    n_done = len(done) + c0 + len(sub)
    rate = (c0 + len(sub)) / max(time.time() - t0, 1e-9)
    eta = (len(todo) - c0 - len(sub)) / max(rate, 1e-9) / 60
    print(f'checkpoint: {n_done:,}/{N_EXPECTED:,} | {rate:,.0f} rows/s | ETA {eta:.1f} min | truncated so far: {n_trunc}')
print(f'FULL RUN DONE. truncated inputs (>{MAX_LEN} tokens): {n_trunc} ({n_trunc / N_EXPECTED:.4%}) - expect ~0 given the 500-char cap')

In [ ]:
# Step 5: consolidate the partial file into finbert_scores.csv with range and sum checks
part = pd.read_csv(PARTIAL)
n_dup = int(part.duplicated('hid').sum())
if n_dup:
    print(f'note: {n_dup} duplicate hids in the partial file (restart overlap) - keeping first')
    part = part.drop_duplicates('hid', keep='first')
part = part.sort_values('hid').reset_index(drop=True)
assert len(part) == N_EXPECTED, f'{len(part):,} rows - some chunks missing, re-run the previous cell'
assert part['hid'].is_unique and part['hid'].min() == 0 and part['hid'].max() == N_EXPECTED - 1
P = part[['p_pos', 'p_neu', 'p_neg']]
assert P.isna().sum().sum() == 0
assert ((P >= 0) & (P <= 1)).all().all()
assert ((P.sum(axis=1) - 1).abs() < 1e-4).all()
part.to_csv(FINAL, index=False)
print(f'wrote {FINAL}: {len(part):,} rows (ordered by hid)')
print('per-column means:', P.mean().round(4).to_dict())
print(part.head(3).to_string(index=False))
print('\nDOWNLOAD finbert_scores.csv -> <repo>/Data/colab/ -> run pipeline/16c_finbert_aggregate_freeze.py')